# Leaf stomatal Medlyn
 

In [ ]:
# | default_exp leaf_stomatal_medlyn

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export
import numpy as np
from plant_hydraulics.leaf_temperature import leaf_temperature
from plant_hydraulics.leaf_boundary_layer import leaf_boundary_layer
from plant_hydraulics.leaf_photosynthesis import leaf_photosynthesis
from plant_hydraulics.leaf_water_potential import leaf_water_potential
from plant_hydraulics.parameter_classes import PhysCon, Leaf, Flux, Atmos

In [ ]:
# | export
def leaf_stomatal_medlyn(
    physcon: PhysCon, atmos: Atmos, leaf: Leaf, flux: Flux
) -> Flux:
    """
    Compute stomatal conductance using the Medlyn et al. (2011) model,
    coupled with leaf energy balance, photosynthesis, and plant hydraulics.

    The Medlyn equation (Medlyn et al. 2011, Eq. 11) is:

        gs = g0 + 1.6 * (1 + g1 / sqrt(D)) * An / cs

    where:

       - gs  = stomatal conductance to water vapor (mol H2O/m2/s)
       - g0  = minimum stomatal conductance (mol H2O/m2/s)
       - g1  = slope parameter (kPa^0.5)
       - D   = vapor pressure deficit at the leaf surface (kPa)
       - An  = net photosynthesis (umol CO2/m2/s)
       - cs  = CO2 concentration at the leaf surface (umol/mol)

    The factor 1.6 converts from CO2 conductance to H2O conductance
    (ratio of molecular diffusivities: D_H2O / D_CO2 = 1.6).

    The coupling problem:
        gs depends on An (more photosynthesis → more stomatal opening)
        An depends on gs (wider stomata → more CO2 → more photosynthesis)
        Tleaf depends on gs (wider stomata → more transpiration → cooler leaf)
        An depends on Tleaf (enzyme kinetics change with temperature)

    Solution approach — fixed-point iteration:

        1. Start with an initial guess for gs
        2. Compute Tleaf from energy balance (leaf_temperature)
        3. Compute An, cs, VPD from photosynthesis (leaf_photosynthesis)
        4. Update gs from Medlyn equation
        5. Repeat until gs converges

    After convergence, a hydraulic safety check is applied:
        If the resulting leaf water potential drops below the minimum
        safe value (leaf.minl_wp), gs is iteratively reduced until
        hydraulic safety is restored. This prevents xylem cavitation.


     The Medlyn equation (Medlyn et al. 2011, Eq. 11):

       gs = g0 + 1.6 * (1 + g1 / sqrt(D)) * An / cs

    where:

    - g0 is the baseline conductance (always present)
    - 1.6 represents the H2O/CO2 diffusivity ratio
    - (1 + g1/sqrt(D)) represents the VPD sensitivity factor:

        - i.e. When D is small (humid): g1/sqrt(D) is large → gs is high
        - i.e. When D is large (dry):   g1/sqrt(D) is small → gs is low
        - g1 controls the magnitude of this VPD response

    - (An / cs) represents the photosynthetic demand factor:
           - High An and low cs → stomata open wide for CO2
           - Low An (shade)     → little need to open stomata

    - Example with g0 = 0.01, g1 = 4.45, D = 1.5 kPa, An = 10, cs = 350:

       gs = 0.01 + 1.6 * (1 + 4.45/sqrt(1.5)) * 10/350
       gs = 0.01 + 1.6 * (1 + 3.633) * 0.02857
       gs = 0.01 + 1.6 * 4.633 * 0.02857
       gs = 0.01 + 0.212
       gs = 0.222 mol H2O/m2/s

    Parameters
    ----------
    physcon : PhysCon
        Physical constants.
    atmos : Atmos
        Atmospheric forcing variables.
    leaf : Leaf
        Leaf parameters including:
        - g0 : float
            Minimum stomatal conductance (mol H2O/m2/s).
        - g1_medlyn : float
            Medlyn slope parameter (kPa^0.5).
        - minl_wp : float
            Minimum leaf water potential (MPa) — the cavitation threshold.
    flux : Flux
        Flux variables. Must have tleaf, qa, psi_leaf, lsc, etc. set.

    Returns
    -------
    flux : Flux
        Updated flux object with converged gs, An, Tleaf, psi_leaf, etc.
    """

    # Compute boundary layer conductances ---------------------------------------

    # WHY first: gbh, gbv, gbc depend on wind speed and leaf size, not on gs.
    # They are needed by leaf_temperature and leaf_photosynthesis.
    flux = leaf_boundary_layer(physcon, atmos, leaf, flux)

    # Set initial guess for gs --------------------------------------------------

    # WHY 0.1: A reasonable starting point for most conditions.
    # Too low (e.g., 0.001) can cause numerical issues in the first
    # photosynthesis calculation. Too high (e.g., 2.0) wastes iterations.
    # The value 0.1 mol/m2/s is typical for an unstressed broadleaf tree.
    flux.gs = 0.1

    # Iteratively solve the gs-An coupling --------------------------------------

    # WHY iterate: gs and An are mutually dependent (circular).
    # We use fixed-point iteration (Picard iteration, Appendix A.5 NEEDX MORE
    # INFO HERE):

    #   gs_{n+1} = Medlyn(An_n, cs_n, VPD_n)

    # This converges because the Medlyn equation is a contraction mapping
    # for reasonable parameter values — each iteration brings gs closer
    # to the fixed point.

    # Safety limit — should converge in 5-15 iterations
    max_iter = 50

    # Convergence tolerance (mol H2O/m2/s)
    # 0.002 is stricter than the 0.004 used in brent_root
    # because we want accurate gs for the hydraulic check
    tol = 0.002

    for each_iteration in range(max_iter):
        # Save previous gs to check convergence
        gs_old = flux.gs

        # Solve leaf energy balance for current gs
        # WHY: Opening stomata changes transpiration, which changes latent
        # heat flux, which changes leaf temperature. We need the correct
        # Tleaf for the photosynthesis temperature adjustments.
        # Example: At gs = 0.3 mol/m2/s, Tleaf might be 33°C.
        #          At gs = 0.1 mol/m2/s, Tleaf might be 36°C (less cooling).
        flux = leaf_temperature(physcon, atmos, leaf, flux)

        # Solve photosynthesis for current gs
        # WHY: We need An, cs, and VPD at the leaf surface.
        # leaf_photosynthesis computes:
        #   - Temperature-adjusted Vcmax, Jmax, Rd, Kc, Ko, Gamma*
        #   - Electron transport rate J
        #   - Rubisco-limited (Ac) and RuBP-limited (Aj) rates
        #   - Net photosynthesis An = Ag - Rd
        #   - Leaf surface CO2 (cs) and intercellular CO2 (ci)
        #   - Leaf surface VPD
        flux = leaf_photosynthesis(physcon, atmos, leaf, flux)

        # Apply the Medlyn equation to compute new gs

        # VPD at leaf surface in kPa (Medlyn equation uses kPa, not Pa)
        # WHY kPa: The g1 parameter has units of kPa^0.5, so D must be in kPa
        # for dimensional consistency.
        # Example: VPD = 1500 Pa → vpd_kpa = 1.5 kPa
        vpd_kpa = flux.vpd / 1000.0

        # Floor VPD at 0.1 kPa to prevent division by zero in sqrt(D)
        # WHY 0.1: At very low VPD (fog, early morning), 1/sqrt(D) → infinity,
        # which would give unrealistically high gs. The floor of 0.1 kPa
        # caps gs at a sensible maximum. This is standard practice in land
        # surface models (e.g., CLM, CABLE).
        vpd_kpa = max(vpd_kpa, 0.1)

        if flux.an > 0:
            # The Medlyn equation (Medlyn et al. 2011, Eq. 11):
            #
            #   gs = g0 + 1.6 * (1 + g1 / sqrt(D)) * An / cs
            #
            # Breaking this down:
            #   g0                  → baseline conductance (always present)
            #   1.6                 → H2O/CO2 diffusivity ratio
            #
            #   (1 + g1/sqrt(D))   → VPD sensitivity factor:
            #       - When D is small (humid): g1/sqrt(D) is large → gs is high
            #       - When D is large (dry):   g1/sqrt(D) is small → gs is low
            #       - g1 controls the magnitude of this VPD response
            #
            #  An / cs             → photosynthetic demand factor:
            #       - High An and low cs → stomata open wide for CO2
            #       - Low An (shade)     → little need to open stomata
            #
            # Example with g0=0.01, g1=4.45, D=1.5 kPa, An=10, cs=350:
            #   gs = 0.01 + 1.6 * (1 + 4.45/sqrt(1.5)) * 10/350
            #   gs = 0.01 + 1.6 * (1 + 3.633) * 0.02857
            #   gs = 0.01 + 1.6 * 4.633 * 0.02857
            #   gs = 0.01 + 0.212
            #   gs = 0.222 mol H2O/m2/s
            flux.gs = (
                leaf.g0
                + 1.6 * (1.0 + leaf.g1_medlyn / np.sqrt(vpd_kpa)) * flux.an / flux.cs
            )
        else:
            # When An <= 0 (e.g., at night, or under very low light where
            # respiration exceeds photosynthesis), the Medlyn equation
            # would give gs < g0 or even negative gs. In this case, set
            # gs to the minimum conductance.
            # WHY: Stomata don't fully close — there's always some residual
            # conductance through the cuticle and incompletely closed guard cells.
            flux.gs = leaf.g0

        # Enforce minimum conductance
        # WHY: Even during the iteration, gs should never drop below g0.
        # This prevents numerical instability in leaf_temperature (which
        # computes gleaf = gs*gbv/(gs+gbv) — if gs = 0, this causes issues).
        flux.gs = max(flux.gs, leaf.g0)

        # Check convergence
        # WHY abs(gs - gs_old): If the change in gs between iterations is
        # smaller than the tolerance, we've found the fixed point.
        # Typical convergence pattern:
        #   Iteration 1: gs = 0.100 → 0.250 (big jump from initial guess)
        #   Iteration 2: gs = 0.250 → 0.220 (overshoot correction)
        #   Iteration 3: gs = 0.220 → 0.225 (settling)
        #   Iteration 4: gs = 0.225 → 0.224 (converged, |Δgs| < 0.002)
        if abs(flux.gs - gs_old) < tol:
            break

    # STEP 4: Hydraulic safety check

    # WHY: The Medlyn equation has no knowledge of plant hydraulics — it
    # only considers the gas-exchange optimality. But in dry soil or tall
    # trees, the transpiration rate implied by the Medlyn gs might pull
    # leaf water potential below the cavitation threshold (leaf.minl_wp).
    #
    # This is equivalent to the "minpsi" check in stomata_efficiency(),
    # but applied as a post-hoc constraint rather than being built into
    # the root-finding process.
    #
    # Approach: If psi_leaf < minl_wp, iteratively reduce gs by 5% until
    # hydraulic safety is restored (or gs hits g0).
    #
    # WHY 5% steps: A gentler reduction than halving. Large jumps can
    # overshoot and give unnecessarily low gs. The iteration is fast
    # because leaf_temperature converges in 3-5 Newton-Raphson steps.

    # First compute fluxes at the converged Medlyn gs
    flux = leaf_boundary_layer(physcon, atmos, leaf, flux)
    flux = leaf_temperature(physcon, atmos, leaf, flux)
    flux = leaf_photosynthesis(physcon, atmos, leaf, flux)

    # Check leaf water potential
    flux = leaf_water_potential(physcon, leaf, flux)

    # If psi_leaf is below the cavitation threshold, reduce gs
    hydraulic_iter = 0
    max_hydraulic_iter = 50

    while flux.psi_leaf < leaf.minl_wp and flux.gs > leaf.g0:
        hydraulic_iter += 1
        if hydraulic_iter > max_hydraulic_iter:
            # Safety exit. Set to minimum conductance
            flux.gs = leaf.g0
            break

        # Reduce gs by 5%
        # WHY 5%: Small enough to find a precise gs, large enough to
        # converge in reasonable iterations. For example, if Medlyn
        # gives gs = 0.3 but hydraulic limit requires gs = 0.2, this
        # takes about 8 iterations: 0.3 → 0.285 → 0.271 → ... → 0.201
        flux.gs = max(flux.gs * 0.95, leaf.g0)

        # Recompute all fluxes at the reduced gs
        flux = leaf_boundary_layer(physcon, atmos, leaf, flux)
        flux = leaf_temperature(physcon, atmos, leaf, flux)
        flux = leaf_photosynthesis(physcon, atmos, leaf, flux)
        flux = leaf_water_potential(physcon, leaf, flux)

    # Final flux computation ----------------------------------------------------

    # After the hydraulic check may have modified gs, ensure
    # all fluxes (An, Tleaf, E, psi_leaf) are consistent with the final gs.
    # If no hydraulic reduction occurred, this is redundant but harmless.
    flux = leaf_boundary_layer(physcon, atmos, leaf, flux)
    flux = leaf_temperature(physcon, atmos, leaf, flux)
    flux = leaf_photosynthesis(physcon, atmos, leaf, flux)
    flux = leaf_water_potential(physcon, leaf, flux)

    return flux

#### Example leaf_stomatal_medlyn()